In [1]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[0])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

from sigpy import mri
import scipy
import pickle
import sigpy as sp
import cupy as cp
import numpy as np
from scipy.io import savemat, loadmat
import twixtools
import matplotlib.pyplot as plt


## My files
import save_data_helpers
import recon_plot_helpers
from ismrm_figs import mvf_utils

### Load ADMM Results

In [2]:
base_dir = '/data/lilianae/ADMM_results/400sp_George/recon_output/'
iter_num = 7

## All options
sweep_dir_high_beta = 'high_beta_tv_z_lam_rho_0.01_beta_0.005_lam_0.007'
sweep_dir_original = 'original_z_rho_0.01_beta_0.0001_lam_0.001'
sweep_dir_strong = 'strong_coupling_z_rho_0.05_beta_0.0005_lam_0.005'

sweep_dir = sweep_dir_original

z_abs, lam_abs, tmp_admm, tmp_inv_admm = mvf_utils.load_admm_output(base_dir, iter_num, sweep_dir)

tmp_inv_admm.shape = (5, 58, 512, 512, 3)
tmp_admm.shape = (5, 58, 512, 512, 3)
z_abs.shape = (5, 58, 512, 512)


### Crop recons + MVFs to desired shape

In [3]:
oshape = (58, 256, 256)

## Motion vectors
cropped_tmp_inv_admm = mvf_utils.crop_gated_mvfs(tmp_inv_admm, oshape)

New array shape = (5, 58, 256, 256, 3)


In [4]:
len(cropped_tmp_inv_admm.shape)

5

In [7]:
import nibabel as nib
import numpy as np


def save_as_numpy(mvf, output_dir, voxel_size=(5.0, 1.172, 1.172)):
    """
    Save 3D MVF as numpy array with specified voxel spacing.
    
    Parameters:
    -----------
    data : numpy.ndarray
        3D array of shape (z, y, x) = (58, 256, 256)
    output_dir : str
        Path where to save the .npy
    voxel_size : tuple
        Voxel dimensions in mm as (z, y, x)
    """
    # Ensure data is in the correct format
    mvf_mm = mvf_utils.convert_mvf_to_mm(mvf, voxel_size)
    
    # Transpose to be same ordering as NiFTI
    mvf_mm_final = np.transpose(mvf_mm, (2,1,0,3))
    mvf_final = np.transpose(mvf, (2,1,0,3))

    # Save to file
    print(f'mvf_mm_final.shape = {mvf_mm_final.shape}')
    # np.save(os.path.join(output_dir, f"mvf_mm_scaled_gate_{gate:02d}.npy"), mvf_mm_final)
    # np.save(os.path.join(output_dir, f"mvf_gate_{gate:02d}.npy"), mvf_final)

    print(f"Saved: {output_dir}")

def save_as_nifti(data, output_path, voxel_size=(5.0, 1.172, 1.172)):
    """
    Save 3D numpy array as NIfTI file with specified voxel spacing.
    
    Parameters:
    -----------
    data : numpy.ndarray
        3D array of shape (z, y, x) = (58, 256, 256)
    output_path : str
        Path where to save the .nii or .nii.gz file
    voxel_size : tuple
        Voxel dimensions in mm as (z, y, x)
    """
    
    # Create affine matrix with voxel spacing
    # The affine matrix encodes voxel size and orientation
    affine = np.diag([voxel_size[2], voxel_size[1], voxel_size[0], 1.0])

    # Create NIfTI image
    nifti_img = nib.Nifti1Image(data.T, affine)  # Transpose for correct orientation
    
    # Save to file
    nib.save(nifti_img, output_path)
    print(f"Saved: {output_path}")


# Save all 5 gates
output_dir = "/home/lilianae/projects/naf_clean/nifti_files/mvfs"
os.makedirs(output_dir, exist_ok=True)

for gate in range(1):
    # Save as NIfTI
    save_as_numpy(cropped_tmp_inv_admm[gate], output_dir, voxel_size=(5.0, 1.172, 1.172))

# print(f"\nAll 5 gates saved to {output_dir}")
print("You can now open these files in ITK-SNAP")

mvf_mm_final.shape = (256, 256, 58, 3)
Saved: /home/lilianae/projects/naf_clean/nifti_files/mvfs
You can now open these files in ITK-SNAP
